# Group 30: all 15 categories

Run the six-method 224px comparison from scratch. Upload only mvtec_anomaly_detection.tar.xz to MyDrive/PRS_Group30, select a Colab GPU runtime, and run in order. No old source ZIP or bottle result is required. This is an adapted full-training-set comparison, not published SOTA reproduction. Keep the checkout and run name unchanged when resuming. Use a new run name after changing code or settings.


In [ ]:
import shutil
import subprocess
import sys
import tarfile
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/PRS_Group30')
DATA_ARCHIVE = BASE / 'mvtec_anomaly_detection.tar.xz'
CATEGORIES = ['bottle', 'cable', 'capsule', 'carpet', 'grid', 'hazelnut', 'leather', 'metal_nut', 'pill', 'screw', 'tile', 'toothbrush', 'transistor', 'wood', 'zipper']  # All categories: bottle, cable, capsule, carpet, grid, hazelnut, leather, metal_nut, pill, screw, tile, toothbrush, transistor, wood, zipper
assert DATA_ARCHIVE.is_file(), f'Upload dataset: {DATA_ARCHIVE}'


## Install the project and check GPU availability

In [ ]:
PROJECT = Path('/content/Visual_Defect_Detection')
# Keep this checkout unchanged when resuming an experiment.
if not (PROJECT / '.git').is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/AndyYeom/Visual_Defect_Detection.git', str(PROJECT)], check=True)
SOURCE_COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT, text=True).strip()
print('Source commit:', SOURCE_COMMIT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(PROJECT) + '[demo,dev]'], check=True)
import torch

print('PyTorch:', torch.__version__)
assert torch.cuda.is_available(), 'Select a GPU runtime and run this cell again.'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=PROJECT, check=True)


## Extract the dataset
Initial extraction may take a while. Training reads images from the Colab local disk rather than Google Drive.

In [ ]:
LOCAL_ARCHIVE = Path('/content/mvtec_anomaly_detection.tar.xz')
DATA_DIR = Path('/content/mvtec_data')
MARKER = DATA_DIR / '.extraction_complete'
if not MARKER.exists():
    DATA_DIR.mkdir(exist_ok=True)
    shutil.copy2(DATA_ARCHIVE, LOCAL_ARCHIVE)
    with tarfile.open(LOCAL_ARCHIVE) as archive:
        archive.extractall(DATA_DIR, filter='data')
    MARKER.touch()
roots = [DATA_DIR] + [p for p in DATA_DIR.iterdir() if p.is_dir()]
matches = [p for p in roots if (p / 'bottle/train/good').is_dir()]
assert len(matches) == 1, 'Cannot identify MVTec AD dataset root'
DATA_ROOT = matches[0]
for category in CATEGORIES:
    assert (DATA_ROOT / category / 'train/good').is_dir(), category
    assert (DATA_ROOT / category / 'test').is_dir(), category
print('Dataset:', DATA_ROOT)


In [ ]:
# Keep RUN_NAME unchanged to resume; use a new name after changing code.
RUN_NAME = 'full15_224_reproduction'
REMOTE = BASE / 'results' / RUN_NAME
REMOTE.mkdir(parents=True, exist_ok=True)
command = [sys.executable, '-u', '-m', 'vdd.suite',
    '--root', str(DATA_ROOT), '--remote', str(REMOTE),
    '--local', str(Path('/content/vdd_runs') / RUN_NAME), '--device', 'cuda']
commit_file = REMOTE / 'source_commit.txt'
if commit_file.exists() and commit_file.read_text().strip() != SOURCE_COMMIT:
    raise ValueError('Restore the saved source commit or choose a new RUN_NAME.')
commit_file.write_text(SOURCE_COMMIT + '\n')
subprocess.run(command, cwd=PROJECT, check=True)


In [ ]:
subprocess.run([sys.executable, str(PROJECT / 'notebooks/analyze_suite.py'), '--run', str(REMOTE), '--root', str(DATA_ROOT)], check=True)
